In [1]:
from tqdm.notebook import tqdm
for i in tqdm(range(100)):
    pass

  0%|          | 0/100 [00:00<?, ?it/s]

In [31]:
import numpy as np

In [2]:
import mlflow

In [3]:

mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("gingo")

2026/03/27 23:46:47 INFO mlflow.tracking.fluent: Experiment with name 'exp1' does not exist. Creating a new experiment.


<Experiment: artifact_location='/mlruns/1', creation_time=1774655207177, experiment_id='1', last_update_time=1774655207177, lifecycle_stage='active', name='exp1', tags={}, workspace='default'>

In [3]:
import polars as pl
data_pl = pl.read_csv(r"../data/train.csv",encoding="shift_jis")

### 前処理ゾーン

In [4]:
# 含水率をlogにして計算しやすくする

data_new = data_pl.with_columns(
    pl.col("含水率").log().alias("含水率_log")
)

In [52]:
#各カラムの差分をとる

exclude_cols = ["sample number", "species number", 
                "樹種", "含水率", "含水率_log"]
num_cols = [c for c in data_new.columns if c not in exclude_cols]

# 横方向の差分カラム名を作成
diff_cols = [f"{num_cols[i]}_diff_{num_cols[i+1]}"
             for i in range(len(num_cols)-1)]

# 横方向の差分を計算して新規カラムとして追加
diff_exprs = [
    (pl.col(num_cols[i+1]) - pl.col(num_cols[i])).alias(diff_cols[i])
    for i in range(len(num_cols)-1)
]

pl_diff_df = data_new.with_columns(diff_exprs)

In [53]:
#varが0/極端に小さい　ものを排除


# 数値列のみ対象
num_cols = [c for c, d in zip(pl_diff_df.columns, pl_diff_df.dtypes) if d in (pl.Float64, pl.Int64)]

# 分散チェックして残す列
valid_cols = [
    c for c in num_cols
    if pl_diff_df.select(pl.col(c).var()).item() != 0
]

# 元の非数値列も残す
other_cols = [c for c in pl_diff_df.columns if c not in num_cols]

# 結合
pl_diff_df_var = pl_diff_df.select(other_cols + valid_cols)

In [84]:
feature_cols = pl_diff_df_var.drop(drop_cols).columns

In [86]:
#show  feature_cols

In [62]:
import polars as pl
import numpy as np
import mlflow
import mlflow.lightgbm
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# ===== MLflow設定 =====
mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("gingo")

# ===== データ =====
pl_df = pl_diff_df_var 

# 目的変数
y = pl_df["含水率_log"].to_numpy()

# 不要列除外
drop_cols = ["含水率", "含水率_log", "樹種", "sample number", "species number"]

X = pl_df.drop(drop_cols).to_numpy()

# ===== 分割 =====
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ===== 学習 =====
params = {
    "n_estimators": 500,
    "learning_rate": 0.05,
    "num_leaves": 64,
    "min_data_in_leaf": 5,
    "n_jobs": -1
}

with mlflow.start_run():

    # パラメータ記録
    mlflow.log_params(params)

    model = LGBMRegressor(**params)
    model.fit(X_train, y_train)

    # 予測
    pred = model.predict(X_valid)

    # RMSE（squared=False使えない環境対応）
    rmse = np.sqrt(mean_squared_error(y_valid, pred))

    # ログ
    mlflow.log_metric("rmse", rmse)

    # モデル保存
    mlflow.lightgbm.log_model(model, "model")

    print("RMSE:", rmse)

[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.226674 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 707985
[LightGBM] [Info] Number of data points in the train set: 1057, number of used features: 3109
[LightGBM] [Info] Start training from score 3.408297


/usr/local/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
2026/03/28 22:32:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5


2026/03/28 22:32:23 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RMSE: 0.100740808419849
🏃 View run calm-bat-566 at: http://mlflow:5000/#/experiments/2/runs/a311f1f89ab742a4aaf882c042db040e
🧪 View experiment at: http://mlflow:5000/#/experiments/2


In [64]:
import mlflow

mlflow.set_tracking_uri("http://mlflow:5000")

# exp1のrun取得
runs = mlflow.search_runs(experiment_names=["gingo"])

# RMSE最小のrunを使う
best_run = runs.sort_values("metrics.rmse").iloc[0]
run_id = best_run["run_id"]

# モデルロード
model = mlflow.lightgbm.load_model(f"runs:/{run_id}/model")

In [73]:
!ls ../data/test.csv

test_data =  pl.read_csv(r"../data/test.csv",encoding="shift_jis")

../data/test.csv


In [76]:
test_data.head(2)

sample number,species number,樹種,9993.76781,9989.9107,9986.05359,9982.19648,9978.33937,9974.48227,9970.62516,9966.76805,9962.91094,9959.05383,9955.19672,9951.33962,9947.48251,9943.6254,9939.76829,9935.91118,9932.05407,9928.19697,9924.33986,9920.48275,9916.62564,9912.76853,9908.91142,9905.05432,9901.19721,9897.3401,9893.48299,9889.62588,9885.76877,9881.91166,9878.05456,9874.19745,9870.34034,9866.48323,…,4138.67729,4134.82018,4130.96307,4127.10596,4123.24886,4119.39175,4115.53464,4111.67753,4107.82042,4103.96331,4100.10621,4096.2491,4092.39199,4088.53488,4084.67777,4080.82066,4076.96356,4073.10645,4069.24934,4065.39223,4061.53512,4057.67801,4053.82091,4049.9638,4046.10669,4042.24958,4038.39247,4034.53536,4030.67826,4026.82115,4022.96404,4019.10693,4015.24982,4011.39271,4007.5356,4003.6785,3999.82139
i64,i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
95,2,"""クスノキ""",0.32877,0.32855,0.32823,0.32792,0.32759,0.32729,0.32713,0.32703,0.32688,0.32674,0.32668,0.32665,0.32654,0.32627,0.32591,0.32565,0.3256,0.32555,0.32529,0.32495,0.32474,0.32461,0.32447,0.32432,0.3241,0.32385,0.32373,0.32354,0.32318,0.32293,0.32281,0.32268,0.32257,0.32246,…,1.27477,1.27645,1.28061,1.286,1.29,1.29307,1.29534,1.29671,1.29833,1.30164,1.30746,1.31474,1.32133,1.32722,1.33457,1.34164,1.34486,1.34568,1.34588,1.34549,1.34356,1.34215,1.34756,1.35521,1.35478,1.34898,1.3474,1.35443,1.36345,1.3647,1.3611,1.35863,1.35918,1.36074,1.35224,1.33495,1.32482
96,2,"""クスノキ""",0.30061,0.30006,0.29978,0.29984,0.29977,0.29945,0.29914,0.29891,0.29878,0.29874,0.29868,0.29861,0.29856,0.29835,0.29795,0.29768,0.29773,0.29783,0.29774,0.29758,0.29743,0.29719,0.29691,0.29666,0.29644,0.29626,0.29612,0.29591,0.29561,0.29545,0.29555,0.29578,0.29592,0.2958,…,1.03124,1.03431,1.03789,1.04243,1.04693,1.05064,1.05287,1.05412,1.05658,1.06155,1.0672,1.07028,1.0707,1.07163,1.07439,1.07874,1.08518,1.09105,1.09306,1.09374,1.09617,1.10041,1.1048,1.10675,1.10801,1.11193,1.11707,1.12196,1.12506,1.12357,1.11911,1.11655,1.11955,1.12671,1.13182,1.1307,1.12377


### 前処理ゾーン

In [89]:
#各カラムの差分をとる

exclude_cols = ["sample number", "species number", 
                "樹種", "含水率", "含水率_log"]

num_cols = [c for c in test_data.columns if c not in exclude_cols]

# 横方向の差分カラム名を作成
diff_cols = [f"{num_cols[i]}_diff_{num_cols[i+1]}"
             for i in range(len(num_cols)-1)]

# 横方向の差分を計算して新規カラムとして追加
diff_exprs = [
    (pl.col(num_cols[i+1]) - pl.col(num_cols[i])).alias(diff_cols[i])
    for i in range(len(num_cols)-1)
]

test_pl_diff_df = test_data.with_columns(diff_exprs)

In [90]:
#del

pl_diff_df_var.head(2)

樹種,sample number,species number,含水率,9993.76781,9989.9107,9986.05359,9982.19648,9978.33937,9974.48227,9970.62516,9966.76805,9962.91094,9959.05383,9955.19672,9951.33962,9947.48251,9943.6254,9939.76829,9935.91118,9932.05407,9928.19697,9924.33986,9920.48275,9916.62564,9912.76853,9908.91142,9905.05432,9901.19721,9897.3401,9893.48299,9889.62588,9885.76877,9881.91166,9878.05456,9874.19745,9870.34034,…,4142.5344_diff_4138.67729,4138.67729_diff_4134.82018,4134.82018_diff_4130.96307,4130.96307_diff_4127.10596,4127.10596_diff_4123.24886,4123.24886_diff_4119.39175,4119.39175_diff_4115.53464,4115.53464_diff_4111.67753,4111.67753_diff_4107.82042,4107.82042_diff_4103.96331,4103.96331_diff_4100.10621,4100.10621_diff_4096.2491,4096.2491_diff_4092.39199,4092.39199_diff_4088.53488,4088.53488_diff_4084.67777,4084.67777_diff_4080.82066,4080.82066_diff_4076.96356,4076.96356_diff_4073.10645,4073.10645_diff_4069.24934,4069.24934_diff_4065.39223,4065.39223_diff_4061.53512,4061.53512_diff_4057.67801,4057.67801_diff_4053.82091,4053.82091_diff_4049.9638,4049.9638_diff_4046.10669,4046.10669_diff_4042.24958,4042.24958_diff_4038.39247,4038.39247_diff_4034.53536,4034.53536_diff_4030.67826,4030.67826_diff_4026.82115,4026.82115_diff_4022.96404,4022.96404_diff_4019.10693,4019.10693_diff_4015.24982,4015.24982_diff_4011.39271,4011.39271_diff_4007.5356,4007.5356_diff_4003.6785,4003.6785_diff_3999.82139
str,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""イチョウ""",1,1,216.129032,0.41485,0.41465,0.41463,0.41476,0.41481,0.4147,0.41452,0.41427,0.41392,0.41364,0.41354,0.41355,0.41352,0.41337,0.41312,0.41284,0.4127,0.41265,0.41257,0.41242,0.4123,0.41226,0.41218,0.41193,0.41159,0.41129,0.41111,0.41102,0.41096,0.41098,0.41109,0.41116,0.41104,…,0.00012,0.00119,0.0037,0.0048,0.00319,0.00056,0.00068,0.00221,0.00109,0.00074,0.00377,0.00516,0.00218,-0.00026,0.00111,0.00303,0.00464,0.0048,0.00038,-0.0022,0.00147,0.00208,-0.002,-0.00176,0.00222,0.00343,0.00312,0.00321,-0.00179,-0.0078,-0.00525,-0.00236,-0.00403,-0.00163,0.00269,0.00267,-0.00135
"""イチョウ""",2,1,210.752688,0.42049,0.4204,0.42049,0.42053,0.42038,0.4201,0.41988,0.41966,0.41942,0.41932,0.41934,0.41934,0.41929,0.41914,0.41887,0.41858,0.41838,0.41829,0.41825,0.4182,0.41809,0.41794,0.4177,0.41753,0.41749,0.41739,0.41728,0.41727,0.41723,0.4171,0.41689,0.41664,0.41651,…,0.00108,0.00158,0.00207,0.00252,0.00331,0.00332,0.00205,0.00081,0.00072,0.00158,0.00297,0.00328,0.00106,-0.00084,0.00041,0.00348,0.00503,0.00338,0.00079,0.0006,0.00463,0.00828,0.00501,-0.00085,-0.0008,0.00289,0.00169,-0.00268,-0.00318,-0.00046,0.0018,-0.00065,-0.00475,-0.00131,0.00434,0.00393,-0.00055


In [92]:
#del

X_test = test_pl_diff_df.select(feature_cols).to_numpy()

In [93]:
#del
X_test

array([[ 0.32877,  0.32855,  0.32823, ..., -0.0085 , -0.01729, -0.01013],
       [ 0.30061,  0.30006,  0.29978, ...,  0.00511, -0.00112, -0.00693],
       [ 0.26048,  0.26033,  0.26   , ..., -0.00644,  0.00145,  0.00593],
       ...,
       [-0.08285, -0.08289, -0.08286, ..., -0.00209, -0.00102, -0.0013 ],
       [-0.08265, -0.08257, -0.08252, ..., -0.00315, -0.00202, -0.00153],
       [-0.0822 , -0.08223, -0.08221, ..., -0.00043, -0.00287, -0.00537]],
      shape=(550, 3109))

### 予測ゾーン

In [94]:
y_pred_log = model.predict(X_test)

[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5


/usr/local/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [95]:
import pandas as pd

y_pred = np.exp(y_pred_log)

submission = pd.DataFrame({
    "id": pl_test["sample number"].to_numpy(),
    "含水率log": y_pred
})

submission.head()

,id,含水率log
0,95,166.113378
1,96,166.651277
2,97,153.954212
3,98,137.396828
4,99,178.414332


In [96]:
submission.to_csv(r"../data/submission.csv",index=False,header = False)

In [97]:
!head ../data/sample_submit.csv

95,50
96,50
97,50
98,50
99,50
100,50
101,50
102,50
103,50
104,50


In [98]:
!head ../data/submission.csv

95,166.11337781799068
96,166.6512773308302
97,153.95421196044404
98,137.396828066121
99,178.41433178232276
100,170.5195446541208
101,155.21145474892003
102,144.03780953678992
103,163.8153437957844
104,140.7844741250766
